# Deliverable 2 - INF201
### Kiki Julia Titia Beumer and Thea Ariansen Gathe

Import the required libraries

In [40]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

## Task 1: File ingestion and regular expressions

1. Write a robust file reader function

In [41]:
def file_reader() -> tuple[str, str]:
    """Read every file under data/2026 and return (file_text_content, encoding_used)."""

    base_dir = Path.cwd() / "data" / "2026"
    files = sorted(p for p in base_dir.rglob("*") if p.is_file())

    texts: list[str] = []
    encodings: list[str] = []

    for f in files:
        raw = f.read_bytes()  # open each file only once

        try:
            texts.append(raw.decode("utf-8-sig"))  # "utf-8-sig" strips a BOM if present
            encodings.append("utf-8")
        except UnicodeDecodeError:
            texts.append(raw.decode("iso-8859-1"))
            encodings.append("iso-8859-1")

    # Preserve order of first appearance, drop duplicates
    encoding_used = ", ".join(dict.fromkeys(encodings))

    return "\n".join(texts), encoding_used

it shows that all files were utf-8 encodinded it is a mix of ascii and utf 

2. Write a log parser function that uses regular expressions

In [42]:
def parse_log(text: str) -> list[dict]:
    """This function extracts valid sales records from log text, ignoring noise lines."""

    # Define what a valid line should look like
    VALID_LINE = re.compile(
        r"""
        ^\[(?P<timestamp>\d{4}-\d{2}-\d{2}\ \d{2}:\d{2}:\d{2})\]   # [2026-10-15 08:30:00]
        \s+\[(?P<register_id>[A-Za-z0-9_]+)\]                      # [POS_01]
        \s+item=(?P<item>.+?)                                      # Latte (spaces allowed)
        \s+price=(?P<price>\d+(?:\.\d+)?)                          # 4.50
        \s+(?P<currency>[A-Z]{3})                                  # EUR, NOK, USD
        \s+\((?P<status>SUCCESS|REFUND|CANCELLED)\)\s*$            # (SUCCESS)
        """,
        re.VERBOSE,
    )

    # Define list to store the valid sales lines 
    valid_sales_lines = []

    # Iterate over each line, store matching lines and remove the noise
    for line in text.splitlines():
        match = VALID_LINE.match(line.strip())

        if not match:
            continue  # skip the noise

        # Store a dictionary of all the named groups from the matches
        record = match.groupdict()

        # Convert the price into a floa
        record["price"] = float(record["price"])

        # Add record to valid_sales_lines list
        valid_sales_lines.append(record)

    return valid_sales_lines

## Task 2: Recursive utility functions

1. Write a recursive directory search function that finds all log files in a nested directory
and returns a list of pathlib.Path objects

In [43]:
def find_log_files(directory: str | Path) -> list[Path]:
    """
    Recursively find all log files in a nested directory and return list of pathlib.Path objects
    """

    # Convert the input path to a pathlib.Path object
    path = Path(directory)
    
    # Initialize empty list to store log files
    log_files = []
    
    # Iterate through items in sorted order to ensure consistent output
    for item in sorted(path.iterdir()):
        # If an item is a directory, recursively search and add to list
        if item.is_dir():
            log_files.extend(find_log_files(item))
        # If an item is a file with a matching extension, add to list
        elif item.is_file() and item.suffix == '.log':
            log_files.append(item)
    
    return log_files

2. Write a recursive function traverses any combination of nested dictionaries and lists found in this file and 
returns a set of all register id strings where "active": true. 

In [44]:
def find_active_registers(node) -> set[str]:
    """
    Recursively collect the ids of all registers with "active": true
    from any combination of nested dicts and lists.
    """
    
    active = set()

    # Traverse through nested dictionary / list 
    if isinstance(node, dict):
        # Check if register dictionary is found
        if 'id' in node and node.get('active') is True:
            active.add(node['id'])
        # Search recursively for all entries in dictionary and add to set
        for value in node.values():
            active |= find_active_registers(value)

    elif isinstance(node, list):
        # If list is found, search recursively for all entries and add to set
        for element in node:
            active |= find_active_registers(element)

    return active


def load_config(file_path) -> dict:
    # Read json file and return dictionary
    with open(Path(file_path), "r", encoding="utf-8") as f:
        return json.load(f)

## Task 3: Functions for cleaning and analysing data using NumPy

1. Write a function for removing price outliers

In [45]:
def remove_price_outliers(valid_sales_lines: list[dict]) -> np.ndarray:
    """Removes price outliers and returns a cleaned 1D array of the prices."""

    prices = np.array([r["price"] for r in valid_sales_lines], dtype=float)
    print(prices.shape)    # (n,)  -> 1D
    print(prices.dtype)    # float64

    # Calculate mean, sd and median
    mean_price = np.mean(prices)
    sd_price = np.std(prices)
    median_price = np.median(prices)

    # Clean the array (replace outliers with the median)
    outliers = np.abs(prices - mean_price) > 3 * sd_price
    prices[outliers] = median_price

    return prices

2. Write a function that computes the element-wise refund ratio

In [46]:
def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    EPS = 1e-6

    refund_ratio = refund_count / (sales_count + EPS)

    return refund_ratio

## Task 4: Data analysis using Pandas

1. Write a function that converts the list into a DataFrame, converts the timestamp to datetime (set as index), and sorts the DataFrame chronologically by timestamp.

In [47]:
def convert_transactions(transactions: list[dict]) -> pd.DataFrame: 
    """
    Converts list of transaction dictionaries into sorted pandas dataframe
    """
    
    # Convert transaction dictionaries into a dataframe
    df = pd.DataFrame(transactions)
    
    # Convert timestamp column to datetime
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    # Set timestamp as index and sort chronologically 
    df = df.set_index('timestamp').sort_index()
    
    return df
    
    

2. Write a function that groups data by item and resamples transactions in 1-hour windows (1h), calculating total price
(sum).

In [48]:
def group_items(transactions: pd.DataFrame) -> pd.DataFrame:
    """
    Takes a dataframe of transactions, groups data by item and resamples
    transactions in 1h windows, calculating total price
    """
    
    return transactions.groupby('item')['price'].resample('1h').sum()

3. Write a function that creates a pivot table displaying total transaction revenue grouped by item across status categories.

In [49]:
def status_total_value(transactions: pd.DataFrame) -> pd.DataFrame:
    """
    Takes a dataframe of transactions, and creates a pivot table displaying
    total transaction revenue grouped by item across status categories
    """
    
    pivot = transactions.pivot_table(
        values="price", 
        index="item", 
        columns="status", 
        aggfunc="sum", 
        fill_value=0
        )
    
    pivot = pivot.reindex(columns=["SUCCESS", "REFUND", "CANCELLED"], fill_value=0)

    return pivot

4. Write a function that has the DataFrame as input and calculates the refund ratio per
item 

In [50]:
def refund_ratio(transactions: pd.DataFrame):
    """
    Takes a dataframe of transactions and calculates the refund ratio per item
    using compute_refund_ratio()
    """
    # Get number of transactions per item and status
    counts = (
        transactions.groupby(["item", "status"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=["SUCCESS", "REFUND", "CANCELLED"], fill_value=0)
    )

    # Make numpy ndarrays of sales and refund count
    sales = counts["SUCCESS"].to_numpy()
    refunds = counts["REFUND"].to_numpy()

    # Return dataframe with sales count, refund count, and ratio per item
    return pd.DataFrame(
        {
            "sales_count": sales,
            "refund_count": refunds,
            "refund_ratio": compute_refund_ratio(sales, refunds),
        },
        index=counts.index,
    )

## Task 5: Create full workflow